# Phase 4 — p4_00 Protocol and Pre-Modelling Audit

Doel: protocol locken, immutable artefacts auditen, event availability contract opstellen, ablation plan exporteren.

In [1]:
from pathlib import Path
import sys


def _add_project_root_to_syspath() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "src").exists() and (candidate / "data_processed").exists():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            return candidate
    raise RuntimeError("Project root with `src/` and `data_processed/` not found in current path hierarchy.")


PROJECT_ROOT = _add_project_root_to_syspath()

from pathlib import Path
import pandas as pd
from IPython.display import display
from src.phase4.protocol import get_phase4_paths, run_pre_modelling_audit

paths = get_phase4_paths()
result = run_pre_modelling_audit(paths=paths)

print('Output map:', paths.results_dir)
print('Protocol:', result['paths']['phase4_protocol_md'])
print('Event contract:', result['paths']['event_contract_csv'])
print('Ablation plan:', result['paths']['ablation_plan_csv'])

print("\nContract checks")
display(result['contract_checks'])

print("\nChecksum audit (first rows)")
display(result['checksum_df'].head(10))

print("\nEvent availability contract (first rows)")
display(result['event_contract'].head(20))

print("\nAblation plan sample")
display(result['ablation_plan'].head(20))


Output map: /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2/data_results/phase4
Protocol: /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2/data_results/phase4/phase4_protocol.md
Event contract: /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2/data_results/phase4/event_feature_availability_contract.csv
Ablation plan: /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2/data_results/phase4/phase4_ablation_plan.csv

Contract checks


,check,result,detail
0,policy_has_no_l_columns,PASS,n_l_columns=0
1,forecast_has_exact_5_selected_l_columns,PASS,n_l_columns=5
2,forecast_excludes_l_validity_flags_from_model_...,PASS,n_l_valid_in_inputs=0
3,immutable_manifest_checksums,PASS,n_checked=11



Checksum audit (first rows)


,artifact,path,expected_sha256,observed_sha256,checksum_ok
0,policy_train.parquet,/Users/emilevandevoorde/Documents/mp_mechelen_...,8c8ead6b8ed69846c1de0d5c876db60e20df4030380ee7...,8c8ead6b8ed69846c1de0d5c876db60e20df4030380ee7...,True
1,policy_holdout_2025.parquet,/Users/emilevandevoorde/Documents/mp_mechelen_...,490eeeb6fd8a3b92c0756b5ee12c8cfde8a443157eaa1e...,490eeeb6fd8a3b92c0756b5ee12c8cfde8a443157eaa1e...,True
2,forecast_train.parquet,/Users/emilevandevoorde/Documents/mp_mechelen_...,c7be991a115a1f2f63b25dcf1a048c1ba648993f141f81...,c7be991a115a1f2f63b25dcf1a048c1ba648993f141f81...,True
3,forecast_holdout_2025.parquet,/Users/emilevandevoorde/Documents/mp_mechelen_...,493d830117ad45281b1a47679c0a8c0ec2204e051eeb4f...,493d830117ad45281b1a47679c0a8c0ec2204e051eeb4f...,True
4,feature_registry.csv,/Users/emilevandevoorde/Documents/mp_mechelen_...,7db132f9d1ef1e49f3985b1704e5bdb67fa663813d3317...,7db132f9d1ef1e49f3985b1704e5bdb67fa663813d3317...,True
5,feature_manifest_policy.json,/Users/emilevandevoorde/Documents/mp_mechelen_...,c3962e21e6da2401555ed7dce135e39e2b885d50841bb2...,c3962e21e6da2401555ed7dce135e39e2b885d50841bb2...,True
6,feature_manifest_forecast.json,/Users/emilevandevoorde/Documents/mp_mechelen_...,b693120448ea6fb149eb662173d92063e64f083f6b16c4...,b693120448ea6fb149eb662173d92063e64f083f6b16c4...,True
7,schema_policy_snapshot.json,/Users/emilevandevoorde/Documents/mp_mechelen_...,f75e4026b2337f9eaa5028af3591eb5f9ab0ee5816986a...,f75e4026b2337f9eaa5028af3591eb5f9ab0ee5816986a...,True
8,schema_forecast_snapshot.json,/Users/emilevandevoorde/Documents/mp_mechelen_...,bf88362229385640e81f0693d5b70959c8dba9ecbd942f...,bf88362229385640e81f0693d5b70959c8dba9ecbd942f...,True
9,data_dictionary_policy.csv,/Users/emilevandevoorde/Documents/mp_mechelen_...,a5f5bfbb79001a1d6adf89cfbfbd527752ab99188a0017...,a5f5bfbb79001a1d6adf89cfbfbd527752ab99188a0017...,True



Event availability contract (first rows)


,feature_name,track_allowed,availability_label,rationale,evidence_source,included_primary_policy,included_primary_forecast
0,e_event_proximity_clip,policy|forecast,ex_post_or_uncertain,Conservative strict rule: timing-proximity sty...,phase4_feature_sets/feature_registry.csv + con...,False,False
1,e_event_scale_is_large,policy|forecast,ex_ante_probable,Event calendar/type signal is usually known in...,phase4_feature_sets/feature_registry.csv + con...,True,True
2,e_event_scale_max,policy|forecast,ex_ante_probable,Event calendar/type signal is usually known in...,phase4_feature_sets/feature_registry.csv + con...,True,True
3,e_event_window_post_24h,policy|forecast,ex_post_or_uncertain,Conservative strict rule: timing-proximity sty...,phase4_feature_sets/feature_registry.csv + con...,False,False
4,e_event_window_pre_24h,policy|forecast,ex_post_or_uncertain,Conservative strict rule: timing-proximity sty...,phase4_feature_sets/feature_registry.csv + con...,False,False
5,e_football_kickoff_hour_cos,policy|forecast,ex_ante_probable,Event calendar/type signal is usually known in...,phase4_feature_sets/feature_registry.csv + con...,True,True
6,e_football_kickoff_hour_sin,policy|forecast,ex_ante_probable,Event calendar/type signal is usually known in...,phase4_feature_sets/feature_registry.csv + con...,True,True
7,e_has_football_kickoff,policy|forecast,ex_ante_probable,Event calendar/type signal is usually known in...,phase4_feature_sets/feature_registry.csv + con...,True,True
8,e_hours_since_event_clip,policy|forecast,ex_post_or_uncertain,Conservative strict rule: timing-proximity sty...,phase4_feature_sets/feature_registry.csv + con...,False,False
9,e_hours_to_event_clip,policy|forecast,ex_post_or_uncertain,Conservative strict rule: timing-proximity sty...,phase4_feature_sets/feature_registry.csv + con...,False,False



Ablation plan sample


,run_id,track,variant,model_family,model_key,feature_set_id,row_filter,seed,primary_metric,cv_mode
0,P0_profile_baseline,policy,P0,baseline,profile_baseline,baseline_profile,none,20260313,mae,fold_safe
1,P0_profile_baseline,policy,P0,baseline,profile_baseline,baseline_profile,none,20260313,mae,fixed_export
2,P1_ridge_a1_0,policy,P1,ridge,ridge_a1_0,policy_ts,none,20260313,mae,fold_safe
3,P1_ridge_a1_0,policy,P1,ridge,ridge_a1_0,policy_ts,none,20260313,mae,fixed_export
4,P1_rf_90_d12_l2,policy,P1,random_forest,rf_90_d12_l2,policy_ts,none,20260313,mae,fold_safe
5,P1_rf_90_d12_l2,policy,P1,random_forest,rf_90_d12_l2,policy_ts,none,20260313,mae,fixed_export
6,P1_xgb_80_d4_lr006,policy,P1,xgboost,xgb_80_d4_lr006,policy_ts,none,20260313,mae,fold_safe
7,P1_xgb_80_d4_lr006,policy,P1,xgboost,xgb_80_d4_lr006,policy_ts,none,20260313,mae,fixed_export
8,P2_ridge_a1_0,policy,P2,ridge,ridge_a1_0,policy_tse_no_parking_id,none,20260313,mae,fold_safe
9,P2_ridge_a1_0,policy,P2,ridge,ridge_a1_0,policy_tse_no_parking_id,none,20260313,mae,fixed_export
